In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib


# Round 2 Reproduction

**Purpose.** Reproduce the Step 7 final-choice and information-acquisition comparisons, one-dimensional parameter sweeps, targeted 50/50 and equal-outcome searches, and approximation diagnostics reported in Round 2.

The historical evidence used 1,200 episodes, 500 VOI samples, common true states, and common observation streams. The notebook calls the shared implementation rather than duplicating model code. Set `RUN = True` only where the full computation should start.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print(PROJECT_ROOT)


In [ ]:
RUN = False
PRESET = "server"
EPISODES = 1200
VOI_SAMPLES = 500
OBSERVATIONS_PER_PERSON = 500
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
OUTPUT_ROOT = "results/round_02_notebook"

print({"run": RUN, "episodes": EPISODES, "voi_samples": VOI_SAMPLES, "max_workers": MAX_WORKERS})


## Step 7, sweeps, DP, and Gauss-Hermite

This command generates final-choice and information-acquisition tables, behavior profiles, approximation-method comparisons, one-dimensional sweeps, DP sensitivity results, and Gauss-Hermite diagnostics.


In [ ]:
main_command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
    "--preset", PRESET,
    "--sections", "step7,sweeps,dp,gh",
    "--episodes", str(EPISODES),
    "--voi-samples", str(VOI_SAMPLES),
    "--common-observations", "on",
    "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
    "--max-workers", str(MAX_WORKERS),
    "--output-dir", OUTPUT_ROOT + "/main",
]
print(" ".join(main_command))
if RUN:
    subprocess.run(main_command, cwd=PROJECT_ROOT, check=True)


## Targeted regime searches

The grids separately search for near-50/50 allocation, symmetric equal-outcome behavior, and equal-outcome behavior that is distinct from equal division.


In [ ]:
GRID_NAMES = ["near_50_50_focused", "equal_outcome_focused", "equal_outcome_distinct_focused"]
GRID_CHUNKS = 32
for grid_name in GRID_NAMES:
    command = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
        "--preset", PRESET,
        "--sections", "regime_grid",
        "--regime-grid", grid_name,
        "--regime-grid-chunks", str(GRID_CHUNKS),
        "--episodes", str(EPISODES),
        "--voi-samples", str(VOI_SAMPLES),
        "--common-observations", "on",
        "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
        "--max-workers", str(MAX_WORKERS),
        "--output-dir", OUTPUT_ROOT + "/" + grid_name,
    ]
    print(" ".join(command))
    if RUN:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Inspect outputs

Missing files are reported without starting or repeating any simulation.


In [ ]:
import pandas as pd

output_root = PROJECT_ROOT / OUTPUT_ROOT
for name in ("step7_final_choice_comparison.csv", "step7_information_acquisition_comparison.csv", "rr_approximation_methods_comparison.csv", "sweep_rr_behavior_profiles.csv"):
    matches = list(output_root.rglob(name)) if output_root.exists() else []
    print(f"{name}: {len(matches)} file(s)")
    if matches:
        display(pd.read_csv(matches[0]).head())
